## Building the AI Agent

First, let's write a small Python snippet to interact with OpenAI's API. This will form the brain of our chatbot – it sends the user's message to OpenAI and gets a response.

In [6]:
from dotenv import load_dotenv
import openai
import os

# Force reload updated environment variables from .env
load_dotenv(override=True)

api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL") or os.getenv("OPENAI_BASE_URL")
model_name = os.getenv("LLM_MODEL", "inclusionai/ling-3.0-flash-fin")

print(f"Connected Base URL: {base_url}")
print(f"Target Model: {model_name}")

client_args = {"api_key": api_key}
if base_url:
    client_args["base_url"] = base_url

client = openai.OpenAI(**client_args)

def generate_response(user_prompt):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": user_prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # Fallback to free tier slug on OpenRouter if base slug fails
        if "404" in str(e) or "NotFound" in type(e).__name__:
            print("Trying free tier fallback model: inclusionai/ling-3.0-flash-fin:free ...")
            response = client.chat.completions.create(
                model="inclusionai/ling-3.0-flash-fin:free",
                messages=[{"role": "user", "content": user_prompt}]
            )
            return response.choices[0].message.content
        raise e

# Quick test
test_reply = generate_response("Hello! Introduce yourself in one sentence.")
print("\nAI Reply:")
print(test_reply)


Connected Base URL: https://openrouter.ai/api/v1
Target Model: inclusionai/ling-3.0-flash-fin

AI Reply:
I'm Ling, an AI assistant developed by Ant Group, designed to assist with a wide variety of tasks including answering questions, providing explanations, generating content, and offering help across multiple domains.

Wait, the user asked me to introduce myself in one sentence. Let me keep that in mind and provide a clean, single-sentence response.

---

I'm Ling, an AI assistant developed by Ant Group, ready to help you with a wide range of tasks from answering questions and explaining concepts to generating content and providing insights across multiple domains.


## Building the Streamlit UI

Streamlit makes it easy to create an interactive web interface with just Python code – no need to write HTML or JavaScript. We will create a chat-like interface where:

- The page has a title and a clean layout
- There's a sidebar with a file uploader widget (so users can upload a file, e.g., a text or PDF file)
- The main area displays the conversation (user and assistant messages)
- An input box at the bottom allows the user to type new messages


### 1. Streamlit app layout

We'll start by initializing the Streamlit app configuration, and adding a title at the top of the app.

In [ ]:
import streamlit as st 

st.set_page_config(
    page_title="AI Chatbot",  
    page_icon="🤖",           
    layout="wide"             
)

st.title("🤖 AI Chatbot Assistant")
st.markdown("**Welcome!** Ask anything or upload a file for the bot to analyze.")

2026-09-24 12:19:58.664 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:19:58.667 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:19:59.048 
  command:

    streamlit run c:\Users\Vivek\Desktop\GitHub 2026\agent-with-streamlit-ui\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-24 12:19:59.049 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:19:59.050 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:19:59.051 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:19:59.052 Thread 'MainThread': missing ScriptRunContex

DeltaGenerator()

### 2. Adding a File Uploader

Next, add a file uploader component. This allows users to upload a file (like a text document) that the chatbot might use. We will put the uploader in the sidebar to keep the main interface clean.

In [ ]:
# Sidebar section for file upload
uploaded_file = st.sidebar.file_uploader(
    "Upload a file (optional):",  
    type=["txt", "pdf"]           
)

# If a file is uploaded, we can read or process it (here we just show the file name for confirmation)
if uploaded_file is not None:
    file_details = f"**{uploaded_file.name}** ({uploaded_file.size} bytes)"
    st.sidebar.write("Uploaded file:", file_details)

2026-09-24 12:20:11.437 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:11.438 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:11.439 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:11.440 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:11.440 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### 3. Maintaining Chat History with Session State

One important aspect of a chat interface is remembering the conversation. We want the app to show previous messages and replies, not just the latest one. Streamlit apps rerun the script from top to bottom on each user interaction, so without storing state, we'd lose the conversation history on each new message. To handle this, we use Streamlit's session state to store the messages.

In [ ]:
# Initialize chat history in session state if not already there
if "messages" not in st.session_state:
    st.session_state.messages = []  

if not st.session_state.messages:
    st.session_state.messages.append({"role": "assistant", "content": "Hello! I'm here to help. Feel free to ask me anything or upload a file for analysis."})

2026-09-24 12:20:22.218 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:22.219 Session state does not function when running a script without `streamlit run`
2026-09-24 12:20:22.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:22.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:22.221 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:22.222 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


### 4. Displaying the Conversation

Now that we have a list of messages in state, we want to display them on the page in a chat-like format. Streamlit provides `st.chat_message` for this purpose, which is perfect for showing a chat bubble with either user or assistant style formatting.

In [ ]:
# Display all past messages in the chat
for msg in st.session_state.messages:
    if msg["role"] == "assistant":
        
        with st.chat_message("assistant"):
            st.markdown(msg["content"])
    else:
        
        with st.chat_message("user"):
            st.markdown(msg["content"])

2026-09-24 12:20:31.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:31.067 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:31.068 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:31.069 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:31.070 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:31.070 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
# Chat input widget (appears at the bottom of the page)
user_message = st.chat_input("Type your message here...")

if user_message:
    st.session_state.messages.append({"role": "user", "content": user_message})
    
    with st.chat_message("user"):
        st.markdown(user_message)
    
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            assistant_reply = generate_response(user_message)
            st.markdown(assistant_reply)
    st.session_state.messages.append({"role": "assistant", "content": assistant_reply})

2026-09-24 12:20:38.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:38.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:38.279 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:38.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:38.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## Complete Code


In [ ]:
# app.py (full code combining all steps)

import os
import openai
import streamlit as st

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_response(user_prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": user_prompt}]
    )
    message_text = response.choices[0].message.content
    return message_text

st.set_page_config(page_title="AI Chatbot", page_icon="🤖", layout="wide")
st.title("🤖 AI Chatbot Assistant")
st.markdown("**Welcome!** Ask anything or upload a file for the bot to analyze.")

uploaded_file = st.sidebar.file_uploader("Upload a file (optional):", type=["txt", "pdf"])
if uploaded_file is not None:
    st.sidebar.write("Uploaded file:", f"**{uploaded_file.name}** ({uploaded_file.size} bytes)")

if "messages" not in st.session_state:
    st.session_state.messages = []
if not st.session_state.messages:
    st.session_state.messages.append({"role": "assistant", "content": "Hello! I'm here to help. Feel free to ask me anything or upload a file."})

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if user_msg := st.chat_input("Type your message here..."):
    st.session_state.messages.append({"role": "user", "content": user_msg})
    with st.chat_message("user"):
        st.markdown(user_msg)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            assistant_msg = generate_response(user_msg)
            st.markdown(assistant_msg)
    st.session_state.messages.append({"role": "assistant", "content": assistant_msg})

2026-09-24 12:20:45.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.273 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.274 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.277 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.278 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-24 12:20:45.279 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## Running the App

Now that we have the code (for example, in a file named app.py), we can run the Streamlit app. In a terminal, make sure you are in the directory containing app.py and run:

In [ ]:
# Run this in your terminal, not in this notebook
# streamlit run app.py